# CHAPTER 10 - Data Aggregation and Group Operations

----

In [105]:
%run "Ch00 - Basic Imports and Set ups.ipynb"

Categorizing a dataset and applying a function to each group, whether an aggregation
or transformation, is often a critical component of a data analysis workflow. After
loading, merging, and preparing a dataset, you may need to compute group statistics
or possibly pivot tables for reporting or visualization purposes. pandas provides a
flexible `groupby` interface, enabling you to slice, dice, and summarize datasets in a
natural way.

One reason for the popularity of relational databases and SQL (which stands for
“structured query language”) is the ease with which data can be joined, filtered, transformed,
and aggregated. However, query languages like SQL are somewhat constrained
in the kinds of group operations that can be performed. As you will see, with
the expressiveness of Python and pandas, we can perform quite complex group operations
by utilizing any function that accepts a pandas object or NumPy array. In this
chapter, you will learn how to:

    • Split a pandas object into pieces using one or more keys (in the form of functions, arrays, or DataFrame column names)
    • Calculate group summary statistics, like count, mean, or standard deviation, or a user-defined function    
    • Apply within-group transformations or other manipulations, like normalization, linear regression, rank, or subset selection
    • Compute pivot tables and cross-tabulations
    • Perform quantile analysis and other statistical group analyses

----

Aggregation of time series data, a special use case of groupby, is
referred to as resampling in this book and will receive separate
treatment in Chapter 11.

## Basic Imports and Set ups

In [2]:
%matplotlib inline
import pandas as pd
import numpy as np
import os
from faker import Faker
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display, HTML
import seaborn as sns

In [3]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 10.1 GroupBy Mechanics

Hadley Wickham, an author of many popular packages for the R programming language,
coined the term split-apply-combine for describing group operations. In the
first stage of the process, data contained in a pandas object, whether a Series, Data‐
Frame, or otherwise, is split into groups based on one or more keys that you provide.
The splitting is performed on a particular axis of an object. For example, a DataFrame
can be grouped on its rows (`axis=0`) or its columns (`axis=1`). Once this is done, a
function is applied to each group, producing a new value. Finally, the results of all
those function applications are combined into a result object. The form of the resulting
object will usually depend on what’s being done to the data. See Figure BELOW for a
mockup of a simple group aggregation.

<img src="images/ch10-1-group.jpg" width="500" alt="Figure 10-1. Illustration of a group aggregation" />

Each grouping key can take many forms, and the keys do not have to be all of the
same type:

    • A list or array of values that is the same length as the axis being grouped
    • A value indicating a column name in a DataFrame
    • A dict or Series giving a correspondence between the values on the axis being grouped and the group names
    • A function to be invoked on the axis index or the individual labels in the index

Note that the latter three methods are shortcuts for producing an array of values to be
used to split up the object. Don’t worry if this all seems abstract. Throughout this
chapter, I will give many examples of all these methods. To get started, here is a small
tabular dataset as a DataFrame:

In [4]:
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randn(5),
                   'data2' : np.random.randn(5)})

In [5]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


Suppose you wanted to compute the mean of the data1 column using the labels from
`key1`. There are a number of ways to do this. One is to access data1 and call groupby
with the column (a Series) at `key1`:

In [6]:
grouped = df['data1'].groupby(df['key1'])

In [7]:
grouped

This grouped variable is now a GroupBy object. It has not actually computed anything
yet except for some intermediate data about the group key `df['key1']`. The idea is
that this object has all of the information needed to then apply some operation to
each of the groups. For example, to compute group means we can call the GroupBy’s
`mean` method:

In [8]:
grouped.mean()

key1
a    0.197324
b    0.666955
Name: data1, dtype: float64

Later, I’ll explain more about what happens when you call `.mean()`. The important
thing here is that the data (a Series) has been aggregated according to the group key,
producing a new Series that is now indexed by the unique values in the `key1` column.

The result index has the name `'key1'` because the DataFrame column `df['key1']`
did.

If instead we had passed multiple arrays as a list, we’d get something different:

In [9]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()

In [10]:
means

key1  key2
a     one    -0.345208
      two     1.282388
b     one     0.193691
      two     1.140219
Name: data1, dtype: float64

Here we grouped the data using two keys, and the resulting Series now has a hierarchical
index consisting of the unique pairs of keys observed:

In [11]:
means.unstack()

key2,one,two
key1,,
a,-0.345208,1.282388
b,0.193691,1.140219


In this example, the group keys are all Series, though they could be any arrays of the
right length:

In [12]:
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])

In [13]:
years = np.array([2005, 2005, 2006, 2005, 2006])

In [14]:
df['data1'].groupby([states, years]).mean()

California  2005    1.282388
            2006    0.193691
Ohio        2005    0.241388
            2006   -0.032972
Name: data1, dtype: float64

Note above , the hierarchical index is build on the group by data set values, not on any column in data frame. 

Frequently the grouping information is found in the same DataFrame as the data you
want to work on. In that case, you can pass column names (whether those are strings,
numbers, or other Python objects) as the group keys:  ( you dont have to use full `dataframe[<column name>]` format)



In [15]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [16]:
df.groupby('key1').mean()

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
df.groupby('key1').mean(numeric_only=True)

##### ✅ Root Cause of This Pandas Error (Very Important Concept)

This error is **NOT a bug in pandas** 🙂
It is a **data-type problem during aggregation.**

---

##### ⭐ What the Error Really Says

The key line is:

```
TypeError: Could not convert string 'onetwoone' to numeric
```

And later:

```
agg function failed [how->mean, dtype->object]
```

This means:

👉 You are doing:

```python
df.groupby('key1').mean()
```

But **at least one column inside `df` is of type `object` (string)**
and pandas is trying to compute **mean on that column.**

---

##### ⭐ Why Pandas Is Trying To Do Mean On Strings

In modern pandas versions:

When you run:

```python
df.groupby('key1').mean()
```

pandas tries to aggregate **ALL columns** unless told otherwise.

If a column contains values like:

```
one
two
one
```

Then internally pandas tries something like:

```
"one" + "two" + "one"
→ "onetwoone"
```

Then it tries to convert that into numeric → 💥 ERROR.

---

##### ⭐ Example That Produces Same Error

```python
df = pd.DataFrame({
    'key1': ['a','a','b'],
    'val1': [1,2,3],
    'val2': ['one','two','one']
})

df.groupby('key1').mean()
```

❌ Boom → same error.

---

##### ✅ Correct Ways To Fix It (Senior DS Level Understanding)

##### ⭐ Fix 1 — Select Only Numeric Columns (Best Practice)

This is the **cleanest real-world solution.**

```python
df.groupby('key1')[['val1']].mean()
```

OR

```python
df.groupby('key1').mean(numeric_only=True)
```

---

##### ⭐ Fix 2 — Convert Column To Numeric (If It Should Be Numeric)

If column is wrongly typed:

```python
df['val2'] = pd.to_numeric(df['val2'], errors='coerce')
```

---

##### ⭐ Fix 3 — Use Different Aggregation For Strings

Sometimes in DS we WANT string aggregation.

Example:

```python
df.groupby('key1').agg({
    'val1': 'mean',
    'val2': 'first'
})
```

---

##### ⭐ VERY Important Data Science Insight (Interview Level)

##### 🔥 GroupBy Aggregations Require Semantic Meaning

You must always ask:

> “Does mean make sense for this column?”

| Column Type     | Valid Aggregation  |
| --------------- | ------------------ |
| numeric         | mean, sum, std     |
| category/string | count, first, mode |
| datetime        | min, max           |

This is a **core Data Science maturity signal.**

---



You may have noticed in the first case `df.groupby('key1').mean()` that there is no
`key2` column in the result. Because `df['key2']` is not numeric data, it is said to be a
nuisance column, which is therefore excluded from the result. By default, all of the
numeric columns are aggregated, though it is possible to filter down to a subset, as
you’ll see soon.
Regardless of the objective in using groupby, a generally useful GroupBy method is
size, which returns a Series containing group sizes:

# 🎯
> Note that, at the time of this notebook (Mar 2026), the nuisance columns can be removed only if you use `numeric_only=True`

Regardless of the objective in using groupby, a generally useful GroupBy method is
`size`, which returns a Series containing group sizes:

In [ ]:
df.groupby(['key1', 'key2']).size()

In [ ]:
df.groupby('key1').size()

Take note that any missing values in a group key will be excluded from the result.

---

### Iterating Over Groups

The GroupBy object supports iteration, generating a sequence of 2-tuples containing
the group name along with the chunk of data. Consider the following:

In [17]:
for name, group in df.groupby('key1'):
    print(name)
    print(group)

a
  key1 key2     data1     data2
0    a  one -0.657443 -0.788235
1    a  two  1.282388  1.573464
4    a  one -0.032972 -0.844257
b
  key1 key2     data1     data2
2    b  one  0.193691 -0.608151
3    b  two  1.140219 -2.649528


In the case of multiple keys, the first element in the tuple will be a tuple of key values:

In [18]:
for (name1,name2), group in df.groupby(['key1', 'key2']) :
    print(name1,name2)
    print(group)

a one
  key1 key2     data1     data2
0    a  one -0.657443 -0.788235
4    a  one -0.032972 -0.844257
a two
  key1 key2     data1     data2
1    a  two  1.282388  1.573464
b one
  key1 key2     data1     data2
2    b  one  0.193691 -0.608151
b two
  key1 key2     data1     data2
3    b  two  1.140219 -2.649528


Of course, you can choose to do whatever you want with the pieces of data. A recipe
you may find useful is computing a dict of the data pieces as a one-liner:

In [19]:
pieces = dict(list(df.groupby('key1')))

In [20]:
pieces

{'a':   key1 key2     data1     data2
 0    a  one -0.657443 -0.788235
 1    a  two  1.282388  1.573464
 4    a  one -0.032972 -0.844257,
 'b':   key1 key2     data1     data2
 2    b  one  0.193691 -0.608151
 3    b  two  1.140219 -2.649528}

In [21]:
pieces['b']

,key1,key2,data1,data2
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528


(a dict of collection of data frames group ed by columns of your interest)

By default `groupby` groups on `axis=0`, but you can group on any of the other axes.
For example, we could group the columns of our example `df` here by `dtype` like so:

In [22]:
df.dtypes

key1      object
key2      object
data1    float64
data2    float64
dtype: object

In [23]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [24]:
grouped = df.groupby(df.dtypes, axis=1)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\521057107.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby(df.dtypes, axis=1)


In [25]:
for dtype, group in grouped:
    print(dtype)
    print(group)

float64
      data1     data2
0 -0.657443 -0.788235
1  1.282388  1.573464
2  0.193691 -0.608151
3  1.140219 -2.649528
4 -0.032972 -0.844257
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


(this can be used to segragate various kind of data types in a frame).

Another way to do the same ( Above one gave deprecation warning).

In [26]:
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group.T)

float64
      data1     data2
0 -0.657443 -0.788235
1  1.282388  1.573464
2  0.193691 -0.608151
3  1.140219 -2.649528
4 -0.032972 -0.844257
object
  key1 key2
0    a  one
1    a  two
2    b  one
3    b  two
4    a  one


##### ✅ Why You Are Getting This Warning

You wrote:

```python
grouped = df.groupby(df.dtypes, axis=1)
```

This means:

👉 **Group columns (NOT rows)** based on their dtype.

Earlier pandas allowed:

```
groupby(..., axis=1)
```

But now this is **deprecated**.

So pandas shows warning:

```
FutureWarning: DataFrame.groupby with axis=1 is deprecated.
Do frame.T.groupby(...) without axis instead.
```

---

##### ✅ Correct Modern Usage (Very Important)

You must **transpose first → then group normally.**

```python
grouped = df.T.groupby(df.dtypes)
```

---

##### ⭐ Why This Works

Original intention:

> Group columns by dtype

But pandas wants **groupby to always work row-wise (axis=0)**

So we convert:

| Step                 | Meaning                                        |
| -------------------- | ---------------------------------------------- |
| `df.T`               | Columns become rows                            |
| `groupby(df.dtypes)` | Now grouping those “rows” (= original columns) |

---

##### ⭐ Full Example (Your Case)

```python
grouped = df.T.groupby(df.dtypes)

for dtype, group in grouped:
    print(dtype)
    print(group)
```

---

##### ⭐ Even Cleaner Senior DS Style

Usually we also store dtype mapping first:

```python
col_types = df.dtypes

grouped = df.T.groupby(col_types)
```

Very readable in production notebooks.

---

##### ⭐ What This Grouping Actually Produces

Your dataframe has:

| Column | dtype  |
| ------ | ------ |
| key1   | object |
| key2   | object |
| data1  | float  |
| data2  | float  |

So grouping result becomes:

```
object → key1, key2
float64 → data1, data2
```

This is extremely useful for:

✅ numeric preprocessing
✅ categorical encoding
✅ automated pipelines

---

##### 🚀 Senior Architect Insight

Modern pandas direction is:

> ❗ Remove axis parameter from many APIs
> ❗ Force consistent row-wise mental model

So future-safe patterns are:

| Old                    | New                 |
| ---------------------- | ------------------- |
| `groupby(..., axis=1)` | `df.T.groupby(...)` |
| `sum(axis=1)`          | still OK            |
| `apply(axis=1)`        | still OK            |

But **groupby(axis=1) → going away.**

---



### Selecting a Column or Subset of Columns

In [27]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [28]:
df['data1']

0   -0.657443
1    1.282388
2    0.193691
3    1.140219
4   -0.032972
Name: data1, dtype: float64

In [29]:
df[['data2']]

,data2
0,-0.788235
1,1.573464
2,-0.608151
3,-2.649528
4,-0.844257


In [31]:
df[['data1', 'data2']]

,data1,data2
0,-0.657443,-0.788235
1,1.282388,1.573464
2,0.193691,-0.608151
3,1.140219,-2.649528
4,-0.032972,-0.844257


Indexing a GroupBy object created from a DataFrame with a column name or array
of column names has the effect of column subsetting for aggregation. This means
that:

In [32]:
df.groupby('key1')['data1']

is same as 

In [33]:
df['data1'].groupby(df['key1'])

In [34]:
df.groupby('key1')[['data2']]

is same as

In [35]:
df[['data2']].groupby(df['key1'])

Especially for large datasets, it may be desirable to aggregate only a few columns. For
example, in the preceding dataset, to compute means for just the `data2` column and
get the result as a DataFrame, we could write:

In [36]:
df.groupby(['key1', 'key2'])[['data2']].mean()

data2
key1 key2          
a    one  -0.816246
     two   1.573464
b    one  -0.608151
     two  -2.649528

The object returned by this indexing operation is a grouped DataFrame if a list or
array is passed or a grouped Series if only a single column name is passed as a scalar:

In [37]:
s_grouped = df.groupby(['key1', 'key2'])['data2']

In [38]:
s_grouped

In [39]:
s_grouped.mean()

key1  key2
a     one    -0.816246
      two     1.573464
b     one    -0.608151
      two    -2.649528
Name: data2, dtype: float64

---

### Grouping with Dicts and Series

Grouping information may exist in a form other than an array. Let’s consider another
example DataFrame:

In [40]:
people = pd.DataFrame(np.random.randn(5, 5),
                      columns=['a', 'b', 'c', 'd', 'e'],
                      index=['Joe', 'Steve', 'Wes', 'Jim', 'Travis'])

In [41]:
people.iloc[2:3, [1, 2]] = np.nan

In [42]:
people

,a,b,c,d,e
Joe,-1.377362,-1.467437,2.371222,-1.773167,-2.125964
Steve,-0.445317,-0.055088,-0.324728,1.526595,0.851903
Wes,-1.134903,NaN,NaN,-0.732003,1.283313
Jim,-0.060313,0.006152,0.116543,-0.387570,0.383990
Travis,-0.188403,1.082705,-0.587520,1.964299,-0.572058


Now, suppose I have a group correspondence for the columns and want to sum
together the columns by group:

In [43]:
mapping = {'a': 'red', 'b': 'red', 'c': 'blue','d': 'blue', 'e': 'red', 'f' : 'orange'}

In [44]:
mapping

{'a': 'red', 'b': 'red', 'c': 'blue', 'd': 'blue', 'e': 'red', 'f': 'orange'}

Now, you could construct an array from this dict to pass to groupby, but instead we
can just pass the dict (I included the key `'f'` to highlight that unused grouping keys
are OK):

In [46]:
by_column = people.groupby(mapping, axis=1)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\3111365124.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  by_column = people.groupby(mapping, axis=1)


In [47]:
by_column.sum()

,blue,red
Joe,0.598056,-4.970763
Steve,1.201867,0.351499
Wes,-0.732003,0.148410
Jim,-0.271026,0.329829
Travis,1.376779,0.322244


The same functionality holds for Series, which can be viewed as a fixed-size mapping:

In [49]:
map_series = pd.Series(mapping)

In [50]:
people.groupby(map_series, axis=1).count()

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\1833935060.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(map_series, axis=1).count()


,blue,red
Joe,2,3
Steve,2,3
Wes,1,2
Jim,2,3
Travis,2,3


### Grouping with Functions

Using Python functions is a more generic way of defining a group mapping compared
with a dict or Series. Any function passed as a group key will be called once per index
value, with the return values being used as the group names. More concretely, consider
the example DataFrame from the previous section, which has people’s first
names as index values. Suppose you wanted to group by the length of the names;
while you could compute an array of string lengths, it’s simpler to just pass the len
function:

In [51]:
people

,a,b,c,d,e
Joe,-1.377362,-1.467437,2.371222,-1.773167,-2.125964
Steve,-0.445317,-0.055088,-0.324728,1.526595,0.851903
Wes,-1.134903,NaN,NaN,-0.732003,1.283313
Jim,-0.060313,0.006152,0.116543,-0.387570,0.383990
Travis,-0.188403,1.082705,-0.587520,1.964299,-0.572058


In [52]:
people.groupby(len).sum()

,a,b,c,d,e
3,-2.572578,-1.461285,2.487765,-2.892739,-0.458661
5,-0.445317,-0.055088,-0.324728,1.526595,0.851903
6,-0.188403,1.082705,-0.587520,1.964299,-0.572058


Mixing functions with arrays, dicts, or Series is not a problem as everything gets converted
to arrays internally:

In [53]:
key_list = ['one', 'one', 'one', 'two', 'two']

In [54]:
people.groupby([len, key_list]).min()

a         b         c         d         e
3 one -1.377362 -1.467437  2.371222 -1.773167 -2.125964
  two -0.060313  0.006152  0.116543 -0.387570  0.383990
5 one -0.445317 -0.055088 -0.324728  1.526595  0.851903
6 two -0.188403  1.082705 -0.587520  1.964299 -0.572058

##### ✅ How `people.groupby([len, key_list]).min()` works

This is actually a **very powerful (and slightly tricky) Pandas feature** 🙂

You are grouping rows using **two grouping keys at the same time**:

1. A **function → `len`**
2. A **list → `key_list`**

Let’s understand step-by-step.

---

## ⭐ Step 1 — Your DataFrame

Index (row labels):

```
Joe
Steve
Wes
Jim
Travis
```

Columns → `a b c d e`

---

## ⭐ Step 2 — First grouping key → `len`

When you pass a **function inside `groupby()`**, Pandas applies that function to the **index values**.

So:

```
len("Joe")     → 3
len("Steve")   → 5
len("Wes")     → 3
len("Jim")     → 3
len("Travis")  → 6
```

So first grouping key becomes:

```
[3, 5, 3, 3, 6]
```

---

## ⭐ Step 3 — Second grouping key → `key_list`

You manually gave:

```
key_list = ['one', 'one', 'one', 'two', 'two']
```

So second grouping key:

```
['one', 'one', 'one', 'two', 'two']
```

---

## ⭐ Step 4 — Pandas combines both keys

Pandas creates **Multi-level grouping (hierarchical grouping)**.

Each row gets a grouping label like:

| Person | len | key_list | Final Group |
| ------ | --- | -------- | ----------- |
| Joe    | 3   | one      | (3, one)    |
| Steve  | 5   | one      | (5, one)    |
| Wes    | 3   | one      | (3, one)    |
| Jim    | 3   | two      | (3, two)    |
| Travis | 6   | two      | (6, two)    |

---

## ⭐ Step 5 — Then `.min()` is applied

Now Pandas computes **column-wise minimum inside each group.**

So effectively:

### Group (3, one) → rows: Joe, Wes

Min is computed column-wise.

### Group (5, one) → row: Steve

Only one row → values stay same.

### Group (3, two) → row: Jim

Same logic.

### Group (6, two) → row: Travis

Same logic.

---

## ⭐ Final Result Structure

You will get a **MultiIndex DataFrame**:

```
             a      b      c      d      e
len key
3   one   ...
    two   ...
5   one   ...
6   two   ...
```

So:

👉 First index level → **length of name**
👉 Second index level → **value from key_list**

---

## 🔥 Very Important Concept (Senior DS Insight)

`groupby()` accepts **ANY of these as grouping keys:**

* column name
* list / array
* Series
* function
* dictionary
* multiple of the above together

This is why:

```python
people.groupby([len, key_list])
```

is valid.

Internally Pandas does:

```
zip(len(index), key_list)
→ build composite group labels
→ split → apply → combine
```

This is classic **Split-Apply-Combine paradigm.**

---

##### ✅ Super Clear Mental Model

Think like this:

> “For each row, compute grouping labels using ALL provided keys,
> then put rows with same labels into one bucket.”

---



## 10.2 Data Aggregation


---

Aggregations refer to any data transformation that produces scalar values from
arrays. The preceding examples have used several of them, including `mean`, `count`,
`min`, and `sum`. You may wonder what is going on when you invoke mean() on a
GroupBy object. Many common aggregations, such as those found in Table 10-1,
have optimized implementations. However, you are not limited to only this set of
methods.

In [56]:
columns = ["Function name", "Description"]

rows = [
["count","Number of non-NA values in the group"],
["sum","Sum of non-NA values"],
["mean","Mean of non-NA values"],
["median","Arithmetic median of non-NA values"],
["std / var","Unbiased (n−1 denominator) standard deviation and variance"],
["min / max","Minimum and maximum of non-NA values"],
["prod","Product of non-NA values"],
["first / last","First and last non-NA values"]
]

render_book_table(
    "Table 10-1. Optimized groupby Methods",
    columns,
    rows
)

Function name,Description
count,Number of non-NA values in the group
sum,Sum of non-NA values
mean,Mean of non-NA values
median,Arithmetic median of non-NA values
std / var,Unbiased (n−1 denominator) standard deviation and variance
min / max,Minimum and maximum of non-NA values
prod,Product of non-NA values
first / last,First and last non-NA values


You can use aggregations of your own devising and additionally call any method that
is also defined on the grouped object. For example, you might recall that `quantile`
computes sample quantiles of a Series or a DataFrame’s columns.

While `quantile` is not explicitly implemented for GroupBy, it is a Series method and
thus available for use. Internally, GroupBy efficiently slices up the Series, calls `piece.quantile(0.9)` for each piece, and then assembles those results together into
the result object:

In [58]:
df

,key1,key2,data1,data2
0,a,one,-0.657443,-0.788235
1,a,two,1.282388,1.573464
2,b,one,0.193691,-0.608151
3,b,two,1.140219,-2.649528
4,a,one,-0.032972,-0.844257


In [59]:
grouped = df.groupby('key1')

In [60]:
grouped['data1'].quantile(0.9)

key1
a    1.019316
b    1.045567
Name: data1, dtype: float64

To use your own aggregation functions, pass any function that aggregates an array to
the aggregate or `agg` method:

In [61]:
def peak_to_peak(arr):
    return arr.max() - arr.min()

In [62]:
grouped.agg(peak_to_peak)

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [63]:
grouped.agg(peak_to_peak, numeric_only=True)

TypeError: peak_to_peak() got an unexpected keyword argument 'numeric_only'

In [64]:
grouped[['data1','data2']].agg(peak_to_peak)

,data1,data2
key1,,
a,1.939831,2.417721
b,0.946528,2.041376


##### ✅ Brief Explanation

You had:

```python
grouped = df.groupby('key1')

def peak_to_peak(arr):
    return arr.max() - arr.min()
```

---

##### ❌ First Method — `grouped.agg(peak_to_peak)`

This **did not work** because Pandas applied the function to **all columns**, including string columns like `key2`.

So it tried:

```
'two' - 'one'
```

String subtraction is not valid → **TypeError.**

---

##### ❌ Second Method — `numeric_only=True`

This also **did not work** because `numeric_only` is **not a Pandas filtering instruction here.**

Instead, Pandas passed it as an argument to your custom function:

```
peak_to_peak(arr, numeric_only=True)
```

But your function does not accept this parameter → **unexpected keyword argument error.**

---

##### ✅ Working Method

Select numeric columns **before aggregation.**

```python
df.groupby('key1')[['data1','data2']].agg(peak_to_peak)
```

Now the function runs only on numeric data → **no string subtraction → works correctly.**

---

✅ **Key Idea:**
Always control which columns go into `groupby().agg()` when using custom functions.




---

You may notice that some methods like `describe` also work, even though they are not
aggregations, strictly speaking:

In [66]:
grouped.describe()

data1                                                              \
     count      mean       std       min       25%       50%       75%   
key1                                                                     
a      3.0  0.197324  0.990209 -0.657443 -0.345208 -0.032972  0.624708   
b      2.0  0.666955  0.669296  0.193691  0.430323  0.666955  0.903587   

               data2                                                    \
           max count      mean       std       min       25%       50%   
key1                                                                     
a     1.282388   3.0 -0.019676  1.379984 -0.844257 -0.816246 -0.788235   
b     1.140219   2.0 -1.628839  1.443471 -2.649528 -2.139184 -1.628839   

                          
           75%       max  
key1                      
a     0.392614  1.573464  
b    -1.118495 -0.608151

>Custom aggregation functions are generally much slower than the
optimized functions found in Table 10-1. This is because there is
some extra overhead (function calls, data rearrangement) in constructing
the intermediate group data chunks.

### Column-Wise and Multiple Function Application

Let’s return to the tipping dataset from earlier examples. After loading it with
`read_csv`, we add a tipping percentage column `tip_pct`:

In [68]:
tips = pd.read_csv('examples/tips.csv')

In [74]:
tips['tip_pct'] = tips['tip'] / tips['total_bill']

In [75]:
tips[:6]

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
5,25.29,4.71,No,Sun,Dinner,4,0.186240


As you’ve already seen, aggregating a Series or all of the columns of a DataFrame is a
matter of using `aggregate` with the desired function or calling a method like `mean` or
`std`. However, you may want to aggregate using a different function depending on the
column, or multiple functions at once. Fortunately, this is possible to do, which I’ll
illustrate through a number of examples.

First, I’ll group the tips by day and smoker:

In [76]:
grouped = tips.groupby(['day', 'smoker'])

In [77]:
grouped_pct = grouped['tip_pct']

In [78]:
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

If you pass a list of functions or function names instead, you get back a DataFrame
with column names taken from the functions:

In [79]:
grouped_pct.agg(['mean', 'std', peak_to_peak])

mean       std  peak_to_peak
day  smoker                                  
Fri  No      0.151650  0.028123      0.067349
     Yes     0.174783  0.051293      0.159925
Sat  No      0.158048  0.039767      0.235193
     Yes     0.147906  0.061375      0.290095
Sun  No      0.160113  0.042347      0.193226
     Yes     0.187250  0.154134      0.644685
Thur No      0.160298  0.038774      0.193350
     Yes     0.163863  0.039389      0.151240

Here we passed a list of aggregation functions to `agg` to evaluate indepedently on the
data groups.

You don’t need to accept the names that GroupBy gives to the columns; notably,
lambda functions have the name `'<lambda>'`, which makes them hard to identify
(you can see for yourself by looking at a function’s `__name__` attribute). Thus, if you
pass a list of `(name, function)` tuples, the first element of each tuple will be used as
the DataFrame column names (you can think of a list of 2-tuples as an ordered
mapping):m

In [83]:
grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\345979284.py:1: FutureWarning: The provided callable <function std at 0x000002988EB10FE0> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  grouped_pct.agg([('foo', 'mean'), ('bar', np.std)])


foo       bar
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

With a DataFrame you have more options, as you can specify a list of functions to
apply to all of the columns or different functions per column. To start, suppose we
wanted to compute the same three statistics for the `tip_pct` and `total_bill`
columns:

In [84]:
functions = ['count', 'mean', 'max']

In [85]:
result = grouped['tip_pct', 'total_bill'].agg(functions)

ValueError: Cannot subset columns with a tuple with more than one element. Use a list instead.

In [87]:
result = grouped[['tip_pct', 'total_bill']].agg(functions)

In [88]:
result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

As you can see, the resulting DataFrame has hierarchical columns, the same as you
would get aggregating each column separately and using `concat` to glue the results
together using the column names as the `keys` argument:

In [90]:
result['tip_pct']

count      mean       max
day  smoker                           
Fri  No          4  0.151650  0.187735
     Yes        15  0.174783  0.263480
Sat  No         45  0.158048  0.291990
     Yes        42  0.147906  0.325733
Sun  No         57  0.160113  0.252672
     Yes        19  0.187250  0.710345
Thur No         45  0.160298  0.266312
     Yes        17  0.163863  0.241255

As before, a list of tuples with custom names can be passed:

In [91]:
ftuples = [('Durchschnitt', 'mean'), ('Abweichung', np.var)]

In [93]:
grouped[['tip_pct', 'total_bill']].agg(ftuples)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\1836388418.py:1: FutureWarning: The provided callable <function var at 0x000002988EB11120> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  grouped[['tip_pct', 'total_bill']].agg(ftuples)


tip_pct              total_bill            
            Durchschnitt Abweichung Durchschnitt  Abweichung
day  smoker                                                 
Fri  No         0.151650   0.000791    18.420000   25.596333
     Yes        0.174783   0.002631    16.813333   82.562438
Sat  No         0.158048   0.001581    19.661778   79.908965
     Yes        0.147906   0.003767    21.276667  101.387535
Sun  No         0.160113   0.001793    20.506667   66.099980
     Yes        0.187250   0.023757    24.120000  109.046044
Thur No         0.160298   0.001503    17.113111   59.625081
     Yes        0.163863   0.001551    19.190588   69.808518

Now, suppose you wanted to apply potentially different functions to one or more of
the columns. To do this, pass a dict to `agg` that contains a mapping of column names
to any of the function specifications listed so far:

In [94]:
grouped.agg({'tip' : np.max, 'size' : 'sum'})

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\683326323.py:1: FutureWarning: The provided callable <function max at 0x000002988EB104A0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  grouped.agg({'tip' : np.max, 'size' : 'sum'})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [95]:
grouped.agg({'tip_pct' : ['min', 'max', 'mean', 'std'],'size' : 'sum'})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

A DataFrame will have hierarchical columns only if multiple functions are applied to
at least one column.

### Returning Aggregated Data Without Row Indexes

In all of the examples up until now, the aggregated data comes back with an index,
potentially hierarchical, composed from the unique group key combinations. Since
this isn’t always desirable, you can disable this behavior in most cases by passing
`as_index=False` to groupby:

In [98]:
tips.groupby(['day', 'smoker'], as_index=False).mean()

TypeError: agg function failed [how->mean,dtype->object]

In [99]:
tips.groupby(['day', 'smoker'], as_index=False).mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


Of course, it’s always possible to obtain the result in this format by calling
`reset_index` on the result. Using the `as_index=False` method avoids some unnecessary
computations.

## 10.3 Apply: General split-apply-combine

The most general-purpose GroupBy method is `apply`, which is the subject of the rest
of this section. As illustrated in Figure , apply splits the object being manipulated
into pieces, invokes the passed function on each piece, and then attempts to concatenate
the pieces together.

<img src="images/ch10-1-group.jpg" width="500" alt="Figure 10-1. Illustration of a group aggregation" />

Returning to the tipping dataset from before, suppose you wanted to select the top
five `tip_pct` values by group. First, write a function that selects the rows with the
largest values in a particular column:

In [109]:
def top(df, n=5, column='tip_pct'):
    return df.sort_values(by=column)[-n:]

In [110]:
top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
232,11.61,3.39,No,Sat,Dinner,2,0.291990
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345


Now, if we group by `smoker`, say, and call `apply` with this function, we get the
following:

In [111]:
tips.groupby('smoker').apply(top)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\1695234324.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby('smoker').apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     88        24.71  5.85     No  Thur   Lunch     2  0.236746
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       232       11.61  3.39     No   Sat  Dinner     2  0.291990
Yes    109       14.31  4.00    Yes   Sat  Dinner     2  0.279525
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       172        7.25  5.15    Yes   Sun  Dinner     2  0.710345

##### ✅ Why you are getting this **FutureWarning** in pandas

When you run:

```python
tips.groupby('smoker').apply(top)
```

pandas is warning you about a **behavior change in future versions.**

---

### 🔴 What is happening internally

Currently (`pandas < future version`):

* `groupby().apply()` **passes the grouping column (`smoker`) also inside each group DataFrame** sent to your function `top`.

So inside `top(group)` you are getting:

```
smoker   total_bill   tip   ...
Yes
Yes
Yes
```

---

### ⚠️ Future pandas behaviour

In future pandas:

* The grouping column (`smoker`) **will NOT be included automatically**
* Only the **non-grouped columns** will be passed.

This is why pandas is warning you now → so your code doesn’t silently break later.

---

### ✅ Proper Fix (Recommended)

Add:

```python
tips.groupby('smoker').apply(top, include_groups=False)
```

✔ This tells pandas:

> “Do NOT include grouping columns when calling apply”

This is **future-proof code** 👍

---

### ✅ Alternative Fix (Explicit column selection — Architect level best practice)

Even better design:

```python
tips.groupby('smoker')[['total_bill', 'tip', 'size']].apply(top)
```

Why this is superior:

* Explicit → readable
* Safer → function receives only required data
* Faster → avoids unnecessary column copying
* Stable → no dependency on pandas internal behavior

---

### 🧠 Senior Data Science Insight

`groupby().apply()` is **very powerful but dangerous**

Use it only when:

* You need full custom logic
* Built-ins like `.agg()`, `.transform()`, `.nlargest()` cannot solve it

Otherwise prefer:

```
groupby().agg()
groupby().transform()
groupby().nlargest()
```

These are:

✅ Vectorized
✅ Faster
✅ More memory efficient
✅ More stable API

---

### 🎯 Example (Typical `top` function case)

If your goal is “top 2 tips per smoker” → architect solution:

```python
tips.sort_values('tip', ascending=False).groupby('smoker').head(2)
```

No `apply()` needed → **clean + fast**

---

If you paste your `top()` function next, I can:

👉 refactor it into **production-grade pandas pattern**
👉 explain performance + memory impact
👉 show multiple alternative implementations

Also note — this aligns with the data science architectural thinking described in your project material  👍


In [112]:
tips.groupby('smoker').apply(top, include_groups=False)

total_bill   tip   day    time  size   tip_pct
smoker                                                    
No     88        24.71  5.85  Thur   Lunch     2  0.236746
       185       20.69  5.00   Sun  Dinner     5  0.241663
       51        10.29  2.60   Sun  Dinner     2  0.252672
       149        7.51  2.00  Thur   Lunch     2  0.266312
       232       11.61  3.39   Sat  Dinner     2  0.291990
Yes    109       14.31  4.00   Sat  Dinner     2  0.279525
       183       23.17  6.50   Sun  Dinner     4  0.280535
       67         3.07  1.00   Sat  Dinner     1  0.325733
       178        9.60  4.00   Sun  Dinner     2  0.416667
       172        7.25  5.15   Sun  Dinner     2  0.710345

What has happened here? The `top` function is called on each row group from the
DataFrame, and then the results are glued together using pandas.`concat`, labeling the
pieces with the group names. The result therefore has a hierarchical index whose
inner level contains index values from the original DataFrame.

If you pass a function to `apply` that takes other arguments or keywords, you can pass
these after the function:

In [122]:
tips.groupby(['smoker', 'day']).apply(top, n=1, column='total_bill', include_groups=False)

total_bill    tip    time  size   tip_pct
smoker day                                                
No     Fri  94        22.75   3.25  Dinner     2  0.142857
       Sat  212       48.33   9.00  Dinner     4  0.186220
       Sun  156       48.17   5.00  Dinner     6  0.103799
       Thur 142       41.19   5.00   Lunch     5  0.121389
Yes    Fri  95        40.17   4.73  Dinner     4  0.117750
       Sat  170       50.81  10.00  Dinner     3  0.196812
       Sun  182       45.35   3.50  Dinner     3  0.077178
       Thur 197       43.11   5.00   Lunch     4  0.115982

>Beyond these basic usage mechanics, getting the most out of apply
may require some creativity. What occurs inside the function
passed is up to you; it only needs to return a pandas object or a
scalar value. The rest of this chapter will mainly consist of examples
showing you how to solve various problems using groupby.
10.3

You may recall that I earlier called `describe` on a GroupBy object:

In [115]:
result = tips.groupby('smoker')['tip_pct'].describe()

In [116]:
result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [117]:
result.unstack('smoker')

       smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

Inside GroupBy, when you invoke a method like describe, it is actually just a shortcut
for:

In [123]:
f = lambda x: x.describe()
grouped.apply(f, include_groups=False)

total_bill       tip  size   tip_pct
day  smoker                                            
Fri  No     count    4.000000  4.000000  4.00  4.000000
            mean    18.420000  2.812500  2.25  0.151650
            std      5.059282  0.898494  0.50  0.028123
            min     12.460000  1.500000  2.00  0.120385
            25%     15.100000  2.625000  2.00  0.137239
...                       ...       ...   ...       ...
Thur Yes    min     10.340000  2.000000  2.00  0.090014
            25%     13.510000  2.000000  2.00  0.148038
            50%     16.470000  2.560000  2.00  0.153846
            75%     19.810000  4.000000  2.00  0.194837
            max     43.110000  5.000000  4.00  0.241255

[64 rows x 4 columns]

### Suppressing the Group Keys

In the preceding examples, you see that the resulting object has a hierarchical index
formed from the group keys along with the indexes of each piece of the original
object. You can disable this by passing `group_keys=False` to groupby:

In [124]:
tips.groupby('smoker', group_keys=False).apply(top, include_groups=False)

,total_bill,tip,day,time,size,tip_pct
88,24.71,5.85,Thur,Lunch,2,0.236746
185,20.69,5.00,Sun,Dinner,5,0.241663
51,10.29,2.60,Sun,Dinner,2,0.252672
149,7.51,2.00,Thur,Lunch,2,0.266312
232,11.61,3.39,Sat,Dinner,2,0.291990
109,14.31,4.00,Sat,Dinner,2,0.279525
183,23.17,6.50,Sun,Dinner,4,0.280535
67,3.07,1.00,Sat,Dinner,1,0.325733
178,9.60,4.00,Sun,Dinner,2,0.416667
172,7.25,5.15,Sun,Dinner,2,0.710345


---

### Quantile and Bucket Analysis

As you may recall from Chapter 8, pandas has some tools, in particular `cut` and `qcut`,
for slicing data up into buckets with bins of your choosing or by sample quantiles.
Combining these functions with `groupby` makes it convenient to perform bucket or
quantile analysis on a dataset. Consider a simple random dataset and an equal-length
bucket categorization using cut:



In [125]:
frame = pd.DataFrame({'data1': np.random.randn(1000),
                      'data2': np.random.randn(1000)})

In [126]:
frame

,data1,data2
0,-0.791567,-0.932087
1,-0.941460,1.018847
2,-0.002612,-2.149812
3,0.379638,-1.778482
4,2.272479,-0.247983
...,...,...
995,-0.188781,1.371683
996,-0.136919,0.321029
997,-0.133926,-1.131054
998,-1.120864,0.829748


In [127]:
quartiles = pd.cut(frame.data1, 4)

In [128]:

quartiles

0       (-1.35, 0.164]
1       (-1.35, 0.164]
2       (-1.35, 0.164]
3       (0.164, 1.679]
4       (1.679, 3.194]
            ...       
995     (-1.35, 0.164]
996     (-1.35, 0.164]
997     (-1.35, 0.164]
998     (-1.35, 0.164]
999    (-2.871, -1.35]
Name: data1, Length: 1000, dtype: category
Categories (4, interval[float64, right]): [(-2.871, -1.35] < (-1.35, 0.164] < (0.164, 1.679] < (1.679, 3.194]]

The Categorical object returned by cut can be passed directly to `groupby`. So we
could compute a set of statistics for the `data2` column like so:

In [132]:
def get_stats(group):
    return {'min': group.min(), 'max': group.max(),'count': group.count(), 'mean': group.mean()}

In [133]:
grouped = frame.data2.groupby(quartiles)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_22632\3611942682.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.data2.groupby(quartiles)


In [134]:
grouped = frame.data2.groupby(quartiles, observed=False)

----

##### ✅ Why this **FutureWarning (observed=False)** is coming

Your code:

```python
frame = pd.DataFrame({'data1': np.random.randn(1000),
                      'data2': np.random.randn(1000)})

quartiles = pd.cut(frame.data1, 4)

grouped = frame.data2.groupby(quartiles)
```

is perfectly correct 👍

But pandas is warning you because **`pd.cut()` creates a Categorical object**, and `groupby()` behaviour with categoricals is changing.

---

##### 🧠 What is happening internally (Very important concept)

`quartiles` is:

```
Categorical
(-3.2, -1.5]
(-1.5, 0.2]
(0.2, 1.9]
(1.9, 3.6]
```

Now when you do:

```python
groupby(quartiles)
```

pandas has 2 possible behaviours:

---

##### 🔴 Old behaviour (current default → `observed=False`)

Groupby will include **ALL possible categories**

Even if some bins have **no data**

Example result index may look like:

```
(-3.2, -1.5]   ✔ data present
(-1.5, 0.2]    ✔ data present
(0.2, 1.9]     ❌ maybe empty
(1.9, 3.6]     ✔ data present
```

So empty groups are also kept.

---

#####🟢 Future behaviour (new default → `observed=True`)

Groupby will include **ONLY categories that actually appear**

Empty bins will be dropped.

This is:

✅ faster
✅ less memory
✅ more logical for analytics

---

##### ✅ Proper Fix (Future-proof)

Just choose behaviour explicitly.

##### ⭐ Recommended (Adopt future pandas behaviour)

```python
grouped = frame.data2.groupby(quartiles, observed=True)
```

---

##### ⭐ If you want current behaviour (keep empty bins)

```python
grouped = frame.data2.groupby(quartiles, observed=False)
```

---

##### 🎯 Architect Level Insight (Very important for Data Science)

This warning appears mainly in workflows like:

* `pd.cut()`
* `pd.qcut()`
* categorical feature engineering
* histogram bin analytics
* cohort analysis
* segmentation modelling

In real production pipelines:

👉 Using `observed=True` is usually better
because:

* avoids fake empty segments
* speeds up aggregations
* reduces dataframe size
* improves plotting clarity

---

##### 🔥 Even Cleaner Pattern (Senior DS style)

Instead of storing intermediate:

```python
grouped = frame.data2.groupby(pd.cut(frame.data1, 4), observed=True)
```

or

```python
frame.groupby(pd.cut(frame.data1, 4), observed=True)['data2']
```

More **pipeline-friendly + readable**

---



In [135]:
grouped.apply(get_stats)

data1                 
(-2.871, -1.35]  min       -2.355420
                 max        2.827117
                 count     86.000000
                 mean      -0.155044
(-1.35, 0.164]   min       -3.464102
                 max        2.803319
                 count    476.000000
                 mean       0.067023
(0.164, 1.679]   min       -3.648976
                 max        2.389452
                 count    383.000000
                 mean      -0.094000
(1.679, 3.194]   min       -2.423380
                 max        1.066732
                 count     55.000000
                 mean      -0.234849
Name: data2, dtype: float64

In [136]:
grouped.apply(get_stats).unstack()

,min,max,count,mean
data1,,,,
"(-2.871, -1.35]",-2.355420,2.827117,86.0,-0.155044
"(-1.35, 0.164]",-3.464102,2.803319,476.0,0.067023
"(0.164, 1.679]",-3.648976,2.389452,383.0,-0.094000
"(1.679, 3.194]",-2.423380,1.066732,55.0,-0.234849


These were equal-length buckets; to compute equal-size buckets based on sample
quantiles, use qcut. I’ll pass `labels=False` to just get quantile numbers:

In [137]:
grouping = pd.qcut(frame.data1, 10, labels=False)

In [138]:
grouped = frame.data2.groupby(grouping)

In [139]:
grouped.apply(get_stats).unstack()

,min,max,count,mean
data1,,,,
0,-2.355420,2.827117,100.0,-0.149442
1,-2.632392,2.803319,100.0,0.206092
2,-2.814519,2.154839,100.0,0.009291
3,-2.503723,2.156035,100.0,-0.045475
4,-3.464102,2.173996,100.0,0.114552
5,-2.167086,2.257218,100.0,0.031106
6,-2.653076,1.906655,100.0,-0.144207
7,-3.648976,1.747461,100.0,-0.325260
8,-2.065883,1.978084,100.0,0.094872


We will take a closer look at pandas’s Categorical type in Chapter 12.

<hr class="red-dot-line">

### Example: Filling Missing Values with Group-Specific Values

When cleaning up missing data, in some cases you will replace data observations
using dropna, but in others you may want to impute (fill in) the null (NA) values
using a fixed value or some value derived from the data. `fillna` is the right tool to
use; for example, here I fill in NA values with the mean:

In [144]:
s = pd.Series(np.random.randn(6))

In [145]:
s[::2] = np.nan

In [148]:
s

0         NaN
1    0.591217
2         NaN
3   -1.800415
4         NaN
5   -1.163213
dtype: float64

In [147]:
s.fillna(s.mean())

0   -0.790804
1    0.591217
2   -0.790804
3   -1.800415
4   -0.790804
5   -1.163213
dtype: float64

Suppose you need the fill value to vary by group. One way to do this is to group the
data and use `apply` with a function that calls `filln`a on each data chunk. Here is
some sample data on US states divided into eastern and western regions:

In [149]:
states = ['Ohio', 'New York', 'Vermont', 'Florida',
          'Oregon', 'Nevada', 'California', 'Idaho']

In [150]:
group_key = ['East'] * 4 + ['West'] * 4

In [153]:
group_key

['East', 'East', 'East', 'East', 'West', 'West', 'West', 'West']

In [151]:
data = pd.Series(np.random.randn(8), index=states)

In [152]:
data

Ohio          0.780855
New York      0.724762
Vermont      -1.344660
Florida      -0.555686
Oregon       -0.721619
Nevada       -0.229669
California    0.187937
Idaho         1.262446
dtype: float64

Let’s set some values in the data to be missing:

In [154]:
data[['Vermont', 'Nevada', 'Idaho']] = np.nan

In [155]:
data

Ohio          0.780855
New York      0.724762
Vermont            NaN
Florida      -0.555686
Oregon       -0.721619
Nevada             NaN
California    0.187937
Idaho              NaN
dtype: float64

In [156]:
data.groupby(group_key).mean()

East    0.316644
West   -0.266841
dtype: float64

We can fill the NA values using the group means like so:

In [157]:
fill_mean = lambda g: g.fillna(g.mean())

In [158]:
data.groupby(group_key).apply(fill_mean)

East  Ohio          0.780855
      New York      0.724762
      Vermont       0.316644
      Florida      -0.555686
West  Oregon       -0.721619
      Nevada       -0.266841
      California    0.187937
      Idaho        -0.266841
dtype: float64

In another case, you might have predefined fill values in your code that vary by
group. Since the groups have a `name` attribute set internally, we can use that:

In [160]:
fill_values = {'East': 0.5, 'West': -1}

In [161]:
fill_func = lambda g: g.fillna(fill_values[g.name])

In [162]:
data.groupby(group_key).apply(fill_func)

East  Ohio          0.780855
      New York      0.724762
      Vermont       0.500000
      Florida      -0.555686
West  Oregon       -0.721619
      Nevada       -1.000000
      California    0.187937
      Idaho        -1.000000
dtype: float64

<hr class="red-dot-line">

### Example: Random Sampling and Permutation

Suppose you wanted to draw a random sample (with or without replacement) from a
large dataset for Monte Carlo simulation purposes or some other application. There
are a number of ways to perform the “draws”; here we use the `sample` method for
Series.

To demonstrate, here’s a way to construct a deck of English-style playing cards:

In [164]:
suits = ['H', 'S', 'C', 'D']

In [165]:
card_val = (list(range(1, 11)) + [10] * 3) * 4

In [166]:
base_names = ['A'] + list(range(2, 11)) + ['J', 'K', 'Q']

In [169]:
card_val

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 10,
 10,
 10,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 10,
 10,
 10,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 10,
 10,
 10,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 10,
 10,
 10]

In [170]:
base_names

['A', 2, 3, 4, 5, 6, 7, 8, 9, 10, 'J', 'K', 'Q']

In [171]:
cards = []
for suit in ['H', 'S', 'C', 'D']:
    cards.extend(str(num) + suit for num in base_names)

In [172]:
deck = pd.Series(card_val, index=cards)

In [174]:
deck

AH      1
2H      2
3H      3
4H      4
5H      5
6H      6
7H      7
8H      8
9H      9
10H    10
JH     10
KH     10
QH     10
AS      1
2S      2
3S      3
4S      4
5S      5
6S      6
7S      7
8S      8
9S      9
10S    10
JS     10
KS     10
QS     10
AC      1
2C      2
3C      3
4C      4
5C      5
6C      6
7C      7
8C      8
9C      9
10C    10
JC     10
KC     10
QC     10
AD      1
2D      2
3D      3
4D      4
5D      5
6D      6
7D      7
8D      8
9D      9
10D    10
JD     10
KD     10
QD     10
dtype: int64

Now, based on what I said before, drawing a hand of five cards from the deck could
be written as:

In [175]:
def draw(deck, n=5):
    return deck.sample(n)

In [176]:
draw(deck)

10D    10
2C      2
8D      8
KH     10
3H      3
dtype: int64

Suppose you wanted two random cards from each suit. Because the suit is the last
character of each card name, we can group based on this and use `apply`:



In [179]:
get_suit = lambda card: card[-1] # last letter is suit

In [180]:
deck.groupby(get_suit).apply(draw, n=2)

C  6C      6
   5C      5
D  QD     10
   2D      2
H  JH     10
   8H      8
S  AS      1
   10S    10
dtype: int64

Alternatively, we could write:    

In [182]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

KC    10
7C     7
9D     9
JD    10
AH     1
6H     6
3S     3
4S     4
dtype: int64

<hr class="red-dot-line">

### Example: Group Weighted Average and Correlation

Under the split-apply-combine paradigm of groupby, operations between columns in
a DataFrame or two Series, such as a group weighted average, are possible. As an
example, take this dataset containing group keys, values, and some weights:

In [185]:
df = pd.DataFrame({'category': ['a', 'a', 'a', 'a',
                                'b', 'b', 'b', 'b'],
                   'data': np.random.randn(8),
                   'weights': np.random.rand(8)})

In [186]:
df

,category,data,weights
0,a,0.805276,0.206369
1,a,0.443726,0.222835
2,a,2.042137,0.312772
3,a,-0.873231,0.195645
4,b,-0.791144,0.710442
5,b,-1.723235,0.849693
6,b,0.416152,0.481748
7,b,2.150652,0.746142


In [187]:
grouped = df.groupby('category')

In [188]:
get_wavg = lambda g: np.average(g['data'], weights=g['weights'])

In [190]:
grouped.apply(get_wavg,include_groups=False )

category
a    0.781704
b   -0.079307
dtype: float64

<hr class="red-dot-line">

#### ✅ The Core Idea — **`groupby().apply()` does NOT decide the shape. Your function does.**

This is the single most important mental model.

> **`apply()` simply takes each group → runs your function → then stitches the returned objects together.**

So the **final output structure depends entirely on what your function returns.**

This is why your two examples behave differently.

(This conceptual clarity is exactly the kind of deep understanding expected while studying data science workflows and pandas behaviour )

---

#### 🔵 Example 1 — Function returns a FULL SERIES (same size as group)

```python
fill_mean = lambda g: g.fillna(g.mean())
data.groupby(group_key).apply(fill_mean)
```

### What happens internally

For group **East**

```
Ohio        0.92
New York   -2.15
Vermont     NaN
Florida    -0.37
```

Function returns:

```
Ohio        0.92
New York   -2.15
Vermont    -0.53   ← filled
Florida    -0.37
```

👉 Notice:

* Returned object = **Series of length 4**
* Index preserved = state names

Same for **West**

Then pandas **concatenates both returned Series**

So final output = **detailed row-level result**

This is conceptually similar to:

✅ **Transform-like behavior**

> “Modify each row inside the group”

---

#### 🔵 Example 2 — Function returns ONE NUMBER per group

```python
get_wavg = lambda g: np.average(g['data'], weights=g['weights'])
grouped.apply(get_wavg)
```

For group **a**

Function returns:

```
0.811643
```

For group **b**

Function returns:

```
-0.122262
```

Now pandas stitches results:

```
category
a   0.811643
b  -0.122262
```

👉 Returned object per group = **scalar**

So final output = **summary per group**

This is conceptually:

✅ **Aggregation-like behavior**

---

#### 🧠 GOLDEN RULE (Very Important)

| Function Return Type      | Final Output Shape | Concept        |
| ------------------------- | ------------------ | -------------- |
| Same length as group      | Detailed rows      | Transform-like |
| Single value              | One row per group  | Aggregation    |
| Different shape DataFrame | Arbitrary          | Fully custom   |

---

#### 🔎 Why pandas behaves this way

Because `apply()` is the **most flexible but least predictable** groupby method.

Internally pandas does:

```
for each group:
    result = function(group)

combine_all_results(result_list)
```

It does NOT enforce:

* fixed row count
* fixed columns
* fixed type

That’s why:

> `apply()` is powerful — but also slower and harder to reason about.

---

#### ⚡ Cleaner Mental Model (Senior DS View)

When using groupby always ask:

👉 **Am I trying to:**

##### 1️⃣ Reduce each group → use `agg`

```
grouped['data'].agg(weighted_avg)
```

##### 2️⃣ Modify each row → use `transform`

```
grouped['data'].transform(lambda g: g.fillna(g.mean()))
```

##### 3️⃣ Do arbitrary custom logic → use `apply`

```
grouped.apply(custom_function)
```

---

#### ⭐ Very Deep Insight (This will save you later)

Your first example is actually **better written as transform**

```
data.groupby(group_key).transform(lambda g: g.fillna(g.mean()))
```

Because:

* output shape = same as input
* pandas optimized path
* faster
* clearer intent

Whereas weighted average is a true aggregation → `apply` or `agg`.

---

#### ✅ Final Intuition

Think like this:

> **`apply()` = “Whatever my function returns… pandas will stack it.”**

Not:

> “groupby decides summary vs detail.”

---

If you want, next I can give you a **very powerful visual mental model diagram (like pandas internal engine view)** — that usually makes `groupby + apply + transform + agg` permanently clear 👍


<hr class="red-dot-line">

As another example, consider a financial dataset originally obtained from Yahoo!
Finance containing end-of-day prices for a few stocks and the S&P 500 index (the SPX
symbol):

In [194]:
close_px = pd.read_csv('examples/stock_px.csv', parse_dates=True,index_col=0)

In [195]:
close_px

,AAPL,MSFT,XOM,SPX
2003-01-02,7.40,21.11,29.22,909.03
2003-01-03,7.45,21.14,29.24,908.59
2003-01-06,7.45,21.52,29.96,929.01
2003-01-07,7.43,21.93,28.95,922.93
2003-01-08,7.28,21.31,28.83,909.93
...,...,...,...,...
2011-10-10,388.81,26.94,76.28,1194.89
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66


In [196]:
close_px.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2214 entries, 2003-01-02 to 2011-10-14
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2214 non-null   float64
 1   MSFT    2214 non-null   float64
 2   XOM     2214 non-null   float64
 3   SPX     2214 non-null   float64
dtypes: float64(4)
memory usage: 86.5 KB


In [197]:
close_px[-4:]

,AAPL,MSFT,XOM,SPX
2011-10-11,400.29,27.00,76.27,1195.54
2011-10-12,402.19,26.96,77.16,1207.25
2011-10-13,408.43,27.18,76.37,1203.66
2011-10-14,422.00,27.27,78.11,1224.58


One task of interest might be to compute a DataFrame consisting of the yearly correlations
of daily returns (computed from percent changes) with SPX. As one way to do
this, we first create a function that computes the pairwise correlation of each column
with the `'SPX'` column:

In [198]:
spx_corr = lambda x: x.corrwith(x['SPX'])

Next, we compute percent change on `close_px` using `pct_change`

In [202]:
rets = close_px.pct_change().dropna()

In [203]:
rets

,AAPL,MSFT,XOM,SPX
2003-01-03,0.006757,0.001421,0.000684,-0.000484
2003-01-06,0.000000,0.017975,0.024624,0.022474
2003-01-07,-0.002685,0.019052,-0.033712,-0.006545
2003-01-08,-0.020188,-0.028272,-0.004145,-0.014086
2003-01-09,0.008242,0.029094,0.021159,0.019386
...,...,...,...,...
2011-10-10,0.051406,0.026286,0.036977,0.034125
2011-10-11,0.029526,0.002227,-0.000131,0.000544
2011-10-12,0.004747,-0.001481,0.011669,0.009795
2011-10-13,0.015515,0.008160,-0.010238,-0.002974


Lastly, we group these percent changes by year, which can be extracted from each row
label with a one-line function that returns the `year` attribute of each `datetime` label:

In [205]:
get_year = lambda x: x.year

In [206]:
by_year = rets.groupby(get_year)

In [209]:
by_year.apply(spx_corr)

,AAPL,MSFT,XOM,SPX
2003,0.541124,0.745174,0.661265,1.0
2004,0.374283,0.588531,0.557742,1.0
2005,0.467540,0.562374,0.631010,1.0
2006,0.428267,0.406126,0.518514,1.0
2007,0.508118,0.658770,0.786264,1.0
2008,0.681434,0.804626,0.828303,1.0
2009,0.707103,0.654902,0.797921,1.0
2010,0.710105,0.730118,0.839057,1.0
2011,0.691931,0.800996,0.859975,1.0


You could also compute inter-column correlations. Here we compute the annual correlation
between Apple and Microsoft:


In [210]:
by_year.apply(lambda g: g['AAPL'].corr(g['MSFT']))

2003    0.480868
2004    0.259024
2005    0.300093
2006    0.161735
2007    0.417738
2008    0.611901
2009    0.432738
2010    0.571946
2011    0.581987
dtype: float64

#### Additional explanation - How Group by Does Grouping on index ( or a function on index)

##### ✅ Why `groupby(get_year)` works but `rets.apply(get_year)` fails

This is a **VERY important pandas + index mental model**.


Let’s go step-by-step like a senior DS debugging this.

---

##### 🔵 Your Data Structure (Very Important)

```python
rets = close_px.pct_change().dropna()
```

So `rets` is:

✅ **DataFrame**

* rows → dates (DatetimeIndex)
* columns → stocks

Example:

```
index (DatetimeIndex) → 2003-01-03
columns → AAPL MSFT XOM SPX
```

---

##### 🔵 What is `get_year` ?

From the book context it is usually:

```python
get_year = lambda x: x.year
```

---

##### ⭐ Now — Why THIS Works

```python
by_year = rets.groupby(get_year)
```

Because:

👉 When you do **DataFrame.groupby(function)**

pandas applies the function to:

> **THE INDEX values — NOT the rows**

Internally pandas does something like:

```
for each index_value in rets.index:
    key = get_year(index_value)
```

Since index_value is:

```
Timestamp('2003-01-03')
```

It HAS:

```
.year
```

So grouping works perfectly.

---

##### 🔵 Why THIS Fails

```python
rets.apply(get_year)
```

Because now pandas behaviour is completely different.

##### Important rule:

> `DataFrame.apply()` works on **columns (Series)** by default.

So pandas is doing:

```
get_year( AAPL column as Series )
get_year( MSFT column as Series )
get_year( XOM column as Series )
```

Each column is:

```
Series of float returns
```

Example:

```
0   0.0023
1   0.0001
2   0.0045
```

Now your function runs:

```
Series.year
```

❌ But Series has NO attribute `year`

Hence:

```
AttributeError: 'Series' object has no attribute 'year'
```

---

##### ⭐ VERY CLEAN MENTAL MODEL

| Operation              | Function receives |
| ---------------------- | ----------------- |
| `groupby(func)`        | index value       |
| `apply()` on DataFrame | column (Series)   |
| `apply(axis=1)`        | row (Series)      |

---

##### 🔵 If You REALLY Want to Extract Year Using Apply

You must operate on index explicitly.

#### Correct ways

##### ✅ Best way (vectorized)

```python
rets.index.year
```

---

##### ✅ Create year column

```python
rets['year'] = rets.index.year
```

---

##### ✅ Or group like this (very clean)

```python
rets.groupby(rets.index.year)
```

This is actually **preferred over lambda**.

Senior DS style 👍

---

##### 🔵 If you want row-wise apply (still not useful here)

```python
rets.apply(lambda row: row.name.year, axis=1)
```

Because:

* row.name = index value (Timestamp)

But this is:

❌ slow
❌ unpythonic
❌ unnecessary

---

##### ⭐ SUPER IMPORTANT Insight (Book Level)

`groupby(func)` is very special.

It is one of the **very few pandas APIs that auto-targets the index.**

Whereas:

* `apply`
* `agg`
* `transform`

operate on **data blocks (Series / rows)**

---

##### ✅ Final Intuition

```
groupby(function)
→ function(index_value)

apply(function)
→ function(series_of_values)
```

That is the whole story.

---


### Example: Group-Wise Linear Regression

In the same theme as the previous example, you can use `groupby` to perform more
complex group-wise statistical analysis, as long as the function returns a pandas
object or scalar value. For example, I can define the following regress function
(using the `statsmodels`econometrics library), which executes an ordinary least
squares (OLS) regression on each chunk of data:

In [220]:
import statsmodels.api as sm
def regress(data, yvar, xvars):
    Y = data[yvar]
    X = data[xvars]
    X['intercept'] = 1.
    #print(type(X))
    #print(X)
    result = sm.OLS(Y, X).fit()
    return result.params

Now, to run a yearly linear regression of AAPL on SPX returns, execute:

In [221]:
by_year.apply(regress, 'AAPL', ['SPX'])

,SPX,intercept
2003,1.195406,0.000710
2004,1.363463,0.004201
2005,1.766415,0.003246
2006,1.645496,0.000080
2007,1.198761,0.003438
2008,0.968016,-0.001110
2009,0.879103,0.002954
2010,1.052608,0.001261
2011,0.806605,0.001514


## 10.4 Pivot Tables and Cross-Tabulation

A pivot table is a data summarization tool frequently found in spreadsheet programs
and other data analysis software. It aggregates a table of data by one or more keys,
arranging the data in a rectangle with some of the group keys along the rows and
some along the columns. Pivot tables in Python with pandas are made possible
through the `groupby` facility described in this chapter combined with reshape operations
utilizing hierarchical indexing. DataFrame has a `pivot_table` method, and
there is also a top-level `pandas.pivot_table` function. In addition to providing a
convenience interface to `groupby`, `pivot_table` can add partial totals, also known as
margins.

Returning to the tipping dataset, suppose you wanted to compute a table of group
means (the default `pivot_table` aggregation type) arranged by `day` and `smoker` on
the rows:

In [223]:
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [228]:
tips.pivot_table(index=['day', 'smoker'])

TypeError: agg function failed [how->mean,dtype->object]

In [229]:
tips.pivot_table(
    values=[ 'total_bill', 'tip', 'size' , 'tip_pct' ],
    index=['day', 'smoker'],
    aggfunc='mean'
)

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

#### Additional Explanation

##### ✅ Core Problem — Mean aggregation failing on mixed dtype

* Your `tips` dataframe has **numeric + object columns**.
* Aggregation like `mean()` fails when pandas tries to compute mean on **object columns** (`smoker`, `day`, `time`).
* Newer pandas versions are stricter → they do not silently ignore non-numeric columns.

---

##### ⭐ Correct Ways to Fix Aggregation Error

**Select only numeric columns (best clarity)**

```python
tips.groupby('day')[['total_bill','tip','size','tip_pct']].mean()
```

**Use numeric_only**

```python
tips.groupby('day').mean(numeric_only=True)
```

**Use column-wise aggregation mapping (real DS pattern)**

```python
tips.groupby('day').agg({
    'total_bill':'mean',
    'tip':'mean',
    'size':'mean',
    'tip_pct':'mean',
    'smoker':'count'
})
```

**Dynamic numeric selection (pipeline friendly)**

```python
num_cols = tips.select_dtypes(include='number').columns
tips.groupby('day')[num_cols].mean()
```

---

##### ✅ Using Pivot Instead of GroupBy

If you want Excel-style summary tables → use **`pivot_table()`**.

You must always specify:

* `values` → numeric column
* `index` → grouping key
* `aggfunc` → aggregation

Example:

```python
tips.pivot_table(
    values='tip_pct',
    index='day',
    aggfunc='mean'
)
```

---

##### ⭐ Pivot with Multiple Group Keys

Mean tip % by **day × smoker**

```python
tips.pivot_table(
    values='tip_pct',
    index='day',
    columns='smoker',
    aggfunc='mean'
)
```

Equivalent to:

```python
tips.groupby(['day','smoker'])['tip_pct'].mean().unstack()
```

---

##### ⭐ Pivot Multiple Numeric Columns

```python
tips.pivot_table(
    values=['tip','total_bill','tip_pct'],
    index='day',
    aggfunc='mean'
)
```

---

##### ⭐ Powerful Reporting Pivot (Real EDA Pattern)

```python
tips.pivot_table(
    values='tip_pct',
    index=['day','time'],
    columns='smoker',
    aggfunc='mean',
    margins=True
)
```

Gives:

* cross-tab summary
* subgroup averages
* grand totals

---

##### ⭐ Very Important Difference

| Function        | Aggregation | Purpose           |
| --------------- | ----------- | ----------------- |
| `pivot()`       | ❌ No        | pure reshape      |
| `pivot_table()` | ✅ Yes       | group + summarize |

---

##### ⭐ Senior Data Science Intuition

* Use **groupby** → when building data pipelines / feature engineering
* Use **pivot_table** → when building summaries / reports / matrix views



<hr class="red-dot-line">

This could have been produced with groupby directly. Now, suppose we want to
aggregate only `tip_pct` and `size`, and additionally group by `time`. I’ll put smoker in
the table columns and day in the rows:

In [231]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],columns='smoker')

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

We could augment this table to include partial totals by passing `margins=True`. This
has the effect of adding `All` row and column labels, with corresponding values being
the group statistics for all the data within a single tier:

In [232]:
tips.pivot_table(['tip_pct', 'size'], index=['time', 'day'],columns='smoker', margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

Here, the `All` values are means without taking into account smoker versus nonsmoker
(the All columns) or any of the two levels of grouping on the rows (the `All`
row).

To use a different aggregation function, pass it to `aggfunc`. For example, `'count'` or
`len` will give you a cross-tabulation (count or frequency) of group sizes:

In [233]:
tips.pivot_table('tip_pct', index=['time', 'smoker'], columns='day',aggfunc=len, margins=True)

day             Fri   Sat   Sun  Thur  All
time   smoker                             
Dinner No       3.0  45.0  57.0   1.0  106
       Yes      9.0  42.0  19.0   NaN   70
Lunch  No       1.0   NaN   NaN  44.0   45
       Yes      6.0   NaN   NaN  17.0   23
All            19.0  87.0  76.0  62.0  244

If some combinations are empty (or otherwise `NA`), you may wish to pass a `fill_value`:

In [234]:
tips.pivot_table('tip_pct', index=['time', 'size', 'smoker'],columns='day', aggfunc='mean', fill_value=0)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
            Yes     0.000000  0.144995  0.152660  0.000000
       4    No      0.000000  0.150096  0.148143  0.000000
            Yes     0.117750  0.124515  0.193370  0.000000
       5    No      0.000000  0.000000  0.206928  0.000000
            Yes     0.000000  0.106572  0.065660  0.000000
       6    No      0.000000  0.000000  0.103799  0.000000
Lunch  1    No      0.000000  0.000000  0.000000  0.181728
            Yes     0.223776  0.000000  0.000000  0.000000
       2    No      0.000000  0.000000  0.000000  0.166005
            Yes     0.181969  0.000000  0.000000  0.158843
       3    No      0.187735  0.000000  0.000000  0.084246
            Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

See Table 10-2 for a summary of `pivot_table` methods.

In [235]:
columns = ["Function name", "Description"]

rows = [
["values","Column name(s) to aggregate; default aggregates all numeric columns"],
["index","Column names or group keys for rows of pivot table"],
["columns","Column names or group keys for columns of pivot table"],
["aggfunc","Aggregation function or list of functions ('mean' by default)"],
["fill_value","Replace missing values in result table"],
["dropna","If True, exclude columns whose entries are all NA"],
["margins","Add row/column subtotals and grand total"]
]

render_book_table(
    "Table 10-2. pivot_table Options",
    columns,
    rows
)

Function name,Description
values,Column name(s) to aggregate; default aggregates all numeric columns
index,Column names or group keys for rows of pivot table
columns,Column names or group keys for columns of pivot table
aggfunc,Aggregation function or list of functions ('mean' by default)
fill_value,Replace missing values in result table
dropna,"If True, exclude columns whose entries are all NA"
margins,Add row/column subtotals and grand total


### Cross-Tabulations: Crosstab


A cross-tabulation (or crosstab for short) is a special case of a pivot table that computes
group frequencies. Here is an example:

In [237]:
data = pd.DataFrame({
    "Sample": range(1, 11),
    "Nationality": [
        "USA", "Japan", "USA", "Japan", "Japan",
        "Japan", "USA", "USA", "Japan", "USA"
    ],
    "Handedness": [
        "Right-handed", "Left-handed", "Right-handed", "Right-handed",
        "Left-handed", "Right-handed", "Right-handed",
        "Left-handed", "Right-handed", "Right-handed"
    ]
})

In [238]:
data

,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


As part of some survey analysis, we might want to summarize this data by nationality
and handedness. You could use `pivot_table` to do this, but the `pandas.crosstab`
function can be more convenient:

In [239]:
pd.crosstab(data.Nationality, data.Handedness, margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


The first two arguments to `crosstab` can each either be an array or Series or a list of
arrays. As in the `tips` data:

In [240]:
pd.crosstab([tips.time, tips.day], tips.smoker, margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244

# 10.5 Conclusion

Mastering pandas’s data grouping tools can help both with data cleaning as well as
modeling or statistical analysis work. In Chapter 14 we will look at several more
example use cases for groupby on real data.
In the next chapter, we turn our attention to time series data.